# Redes Convolucionales: filtros, feature maps y encoding

Objetivo: entender que hace un filtro convolucional, como la red transforma una imagen en
una representación abstracta (encoding), y cuando usar CNNs.

In [1]:
import os, pathlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import convolve
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
IMG = pathlib.Path('images')
IMG.mkdir(exist_ok=True)
print('Setup OK')
print(f'numpy: {np.__version__}')

Setup OK
numpy: 1.26.4


## 1. Que es un filtro y por que funciona

En una CNN, un filtro (o kernel) es una pequeña matriz de pesos (típicamente 3x3) que se
desplaza sobre la imagen multiplicando cada parche por sus pesos y sumando (producto escalar).
El resultado es un único número: la 'respuesta' del filtro en esa posición.

La intuición: un filtro diseñado para detectar bordes verticales da respuesta alta donde hay
un borde vertical y baja donde no. La red aprende estos filtros automáticamente durante el
entrenamiento.

In [2]:
# Crear imagen sintetica 64x64 con figuras geometricas
img = np.zeros((64, 64), dtype=float)
img[10:30, 10:50] = 1.0   # rectangulo horizontal claro
img[35:55, 20:40] = 0.7   # rectangulo oscuro
# Anadir circulo aproximado
for i in range(64):
    for j in range(64):
        if (i-48)**2 + (j-48)**2 < 10**2:
            img[i, j] = 0.5
img += np.random.normal(0, 0.05, img.shape)  # ruido
img = np.clip(img, 0, 1)

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap='gray')
plt.title('Imagen sintetica 64x64')
plt.axis('off')
plt.tight_layout()
plt.show()
print('Imagen sintetica creada: shape =', img.shape)

Imagen sintetica creada: shape = (64, 64)


In [3]:
from scipy.ndimage import convolve

# Definir filtros clasicos
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float)
blur    = np.ones((3, 3), dtype=float) / 9.0

# Aplicar filtros
feat_x   = convolve(img, sobel_x)
feat_y   = convolve(img, sobel_y)
feat_blur = convolve(img, blur)
feat_grad = np.sqrt(feat_x**2 + feat_y**2)
feat_sharp = img - feat_blur

imagenes = [
    (img,        'Original',                  'gray'),
    (feat_x,     'Sobel X (bordes vert.)',     'RdBu'),
    (feat_y,     'Sobel Y (bordes horiz.)',    'RdBu'),
    (feat_blur,  'Suavizado 3x3 (media)',      'gray'),
    (feat_grad,  'Gradiente total (mag.)',     'hot'),
    (feat_sharp, 'Realce de bordes (sharp.)', 'RdBu'),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (imagen, titulo, cmap) in zip(axes.ravel(), imagenes):
    ax.imshow(imagen, cmap=cmap)
    ax.set_title(titulo, fontsize=10)
    ax.axis('off')

plt.suptitle('Filtros convolucionales: cada filtro detecta un patron diferente',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('images/B05C_fig01.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05C_fig01.png')

Guardado: images/B05C_fig01.png


> Antes de seguir: el filtro Sobel X da respuesta alta en bordes verticales.
> Como disenarias un filtro 3x3 que detecte lineas diagonales
> (de arriba-izquierda a abajo-derecha)?

<details>
<summary>Orientacion para el instructor</summary>

Respuesta: kernel con valores altos en la diagonal principal y negativos en la anti-diagonal,
como [[-1,-1,2],[-1,2,-1],[2,-1,-1]]. El filtro se 'parece' al patron que detecta:
valores positivos donde la diagonal tiene mas intensidad, negativos donde no.
Senal de comprension: el alumno entiende que el filtro es una plantilla del patron buscado.

</details>

## 2. La operación de convolucion paso a paso

La ventana 3x3 se desplaza sobre la imagen con stride=1. En cada posicion:
1. Se extrae el parche 3x3 de la imagen
2. Se multiplica elemento a elemento por el filtro
3. Se suman todos los valores -> un único número en el feature map
4. La ventana avanza una posición (stride)

El resultado es un **mapa de características (feature map)**: una imagen mas pequeña donde
cada pixel indica 'cuanto se activa este filtro aqui'.

In [4]:
# Animacion estatica: 6 pasos de la convolucion sobre imagen 6x6
img_small = np.array([
    [3, 0, 1, 2, 7, 4],
    [1, 5, 8, 9, 3, 1],
    [2, 7, 2, 5, 1, 3],
    [0, 1, 3, 1, 7, 8],
    [4, 2, 1, 6, 2, 8],
    [2, 4, 5, 2, 3, 9],
], dtype=float)

kernel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

# Posiciones de la ventana (fila, col) de la esquina sup-izq
posiciones = [(0,0), (0,1), (0,2), (1,0), (1,2), (2,1)]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (r, c) in zip(axes.ravel(), posiciones):
    # Mostrar imagen con overlay
    ax.imshow(img_small, cmap='Blues', vmin=0, vmax=9, alpha=0.7)
    # Resaltar ventana con borde rojo
    rect = plt.Rectangle((c-0.5, r-0.5), 3, 3,
                          linewidth=2.5, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    # Calcular resultado parcial
    parche = img_small[r:r+3, c:c+3]
    resultado = float(np.sum(parche * kernel))
    # Anotar valores del parche
    for i in range(3):
        for j in range(3):
            ax.text(c+j, r+i, f'{int(img_small[r+i, c+j])}',
                    ha='center', va='center', fontsize=9,
                    color='darkred', fontweight='bold')
    ax.set_title(f'Ventana en ({r},{c})\nResultado = {resultado:.1f}',
                 fontsize=9)
    # Mostrar filtro en esquina
    ax.text(5.4, 5.2, f'K=\n[[-1,0,1]\n[-2,0,2]\n[-1,0,1]]',
            fontsize=6, color='green', ha='right', va='bottom')
    ax.set_xlim(-0.5, 5.5)
    ax.set_ylim(5.5, -0.5)
    ax.set_xticks(range(6))
    ax.set_yticks(range(6))
    ax.tick_params(labelsize=7)

plt.suptitle('Convolucion paso a paso: la ventana se desplaza sobre la imagen',
             fontsize=12)
plt.tight_layout()
plt.savefig('images/B05C_fig02.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05C_fig02.png')

Guardado: images/B05C_fig02.png


## 3. Feature maps y ReLU

Despues de la convolucion, se aplica ReLU (max(0, x)):
- Activa solo las zonas donde el filtro 'reconocio' su patron
- Los valores negativos (donde NO hay el patron) se eliminan

Esto produce mapas de características binarios o graduados: presente / ausente / intensidad.

In [5]:
# Pipeline: imagen -> convolucion -> ReLU -> pooling
feature_map = convolve(img, sobel_x)
feature_relu = np.maximum(0, feature_map)

# Max pooling 2x2 manual con numpy
h, w = feature_relu.shape
h2, w2 = h // 2, w // 2
feature_pool = feature_relu[:h2*2, :w2*2].reshape(h2, 2, w2, 2).max(axis=(1, 3))

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('1. Imagen original\n(64x64)', fontsize=10)

im1 = axes[1].imshow(feature_map, cmap='RdBu')
axes[1].set_title('2. Feature map\n(antes de ReLU, valores -/+)', fontsize=10)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(feature_relu, cmap='Blues')
axes[2].set_title('3. Tras ReLU\n(solo positivos)', fontsize=10)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

im3 = axes[3].imshow(feature_pool, cmap='Blues')
axes[3].set_title('4. Max Pooling 2x2\n(32x32, reduccion espacial)', fontsize=10)
plt.colorbar(im3, ax=axes[3], fraction=0.046)

for ax in axes:
    ax.axis('off')

plt.suptitle('Pipeline: imagen -> convolucion -> ReLU -> pooling', fontsize=12)
plt.tight_layout()
plt.savefig('images/B05C_fig03.png', dpi=120, bbox_inches='tight')
plt.close()
print('Shapes:')
print(f'  original:     {img.shape}')
print(f'  feature_map:  {feature_map.shape}')
print(f'  relu:         {feature_relu.shape}')
print(f'  pooling:      {feature_pool.shape}')
print('Guardado: images/B05C_fig03.png')

Shapes:
  original:     (64, 64)
  feature_map:  (64, 64)
  relu:         (64, 64)
  pooling:      (32, 32)
Guardado: images/B05C_fig03.png


## 4. Encoding: de pixeles a conceptos

El encoding es el proceso por el que una CNN transforma la imagen original
(H x W x 3 pixeles de color) en un vector de características abstractas:

```
Capa 1 (Conv+Pool): 64x64x1  -> 32x32x8   (8 detectores de bordes)
Capa 2 (Conv+Pool): 32x32x8  -> 16x16x16  (16 detectores de formas)
Capa 3 (Conv+Pool): 16x16x16 -> 8x8x32    (32 detectores de patrones)
Flatten:            8x8x32   -> vector de 2048
Dense:              2048     -> 10 clases
```

Las primeras capas detectan bordes y texturas. Las capas profundas detectan conceptos
(ojo, rueda, cara). El vector intermedio (el encoding) puede usarse para comparar imagenes,
hacer transfer learning, o como features para otro modelo.

In [6]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.axis('off')

# Definir bloques: (x_centro, etiqueta_top, etiqueta_bot, color)
bloques = [
    (1.2,  'Input',          '64x64x1',   '#455A64'),
    (3.2,  'Conv+Pool 1',    '32x32x8',   '#1565C0'),
    (5.4,  'Conv+Pool 2',    '16x16x16',  '#1565C0'),
    (7.6,  'Conv+Pool 3',    '8x8x32',    '#1565C0'),
    (9.8,  'Flatten',        'vector 2048','#E65100'),
    (11.8, 'Dense 512',      '512',        '#6A1B9A'),
    (13.8, 'Softmax',        '10 clases',  '#B71C1C'),
]

bw = 1.6   # ancho del bloque
bh = 1.4   # alto del bloque
by = 1.8   # coordenada y del centro del bloque

for x, top, bot, color in bloques:
    bbox = FancyBboxPatch((x - bw/2, by - bh/2), bw, bh,
                          boxstyle='round,pad=0.08',
                          facecolor=color, edgecolor='white',
                          linewidth=1.5, alpha=0.88)
    ax.add_patch(bbox)
    ax.text(x, by + 0.25, top, ha='center', va='center',
            fontsize=8.5, color='white', fontweight='bold')
    ax.text(x, by - 0.28, bot, ha='center', va='center',
            fontsize=8, color='#e0e0e0')

# Flechas entre bloques
xs = [b[0] for b in bloques]
for i in range(len(xs)-1):
    x_start = xs[i] + bw/2
    x_end   = xs[i+1] - bw/2
    ax.annotate('', xy=(x_end, by), xytext=(x_start, by),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.8))

# Anotacion encoding
ax.annotate('encoding =\nrepresentacion\naprendida',
            xy=(9.8, by + bh/2), xytext=(9.8, 4.1),
            fontsize=8, color='#E65100', ha='center',
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.2))

# Leyenda de colores
leyenda = [
    mpatches.Patch(color='#1565C0', label='Conv + MaxPool'),
    mpatches.Patch(color='#E65100', label='Flatten'),
    mpatches.Patch(color='#6A1B9A', label='Dense (FC)'),
    mpatches.Patch(color='#B71C1C', label='Softmax -> clase'),
]
ax.legend(handles=leyenda, loc='upper left', fontsize=8,
          framealpha=0.85, ncol=4)

ax.set_title('Encoding CNN: de pixeles a vector de caracteristicas abstractas',
             fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('images/B05C_fig04.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05C_fig04.png')

Guardado: images/B05C_fig04.png


## 5. Por que funciona: invarianza traslacional

La clave del poder de las CNNs es que **el mismo filtro se aplica en todas las posiciones**
de la imagen. Esto significa:
- Un detector de ojos funciona igual si el ojo esta en la esquina o en el centro
- La red no necesita aprender un detector diferente para cada posicion
- Reducción drástica de parámetros: un filtro 3x3 tiene solo 9 parámetros, pero se aplica
  en miles de posiciones

Comparación: una red densa necesita un parámetro por conexión (millones);
una CNN reutiliza los mismos filtros (miles).

In [7]:
# Tabla comparativa CNN vs MLP
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

columnas = ['Caracteristica', 'MLP (red densa)', 'CNN']
filas = [
    ['Parametros para imagen 64x64', '4096 por neurona oculta', '9 por filtro (compartido)'],
    ['Invarianza a posicion',         'No',                      'Si (weight sharing)'],
    ['Eficiencia con imgs grandes',   'Muy mala (escala cuadrat.)','Buena (escala lineal)'],
    ['Datos necesarios',              'Muchos',                  'Menos (transfer learning)'],
    ['Interpretabilidad',             'Baja',                    'Media (filtros visibles)'],
]

colores_col = ['#CFD8DC', '#BBDEFB', '#C8E6C9']
colores_filas = []
for i in range(len(filas)):
    colores_filas.append(['#ECEFF1' if i % 2 == 0 else '#F5F5F5'] * 3)

tabla = ax.table(
    cellText=filas,
    colLabels=columnas,
    cellLoc='center',
    loc='center',
    cellColours=colores_filas,
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.2, 2.1)

# Colorear cabecera
for j, color in enumerate(colores_col):
    tabla[0, j].set_facecolor(color)
    tabla[0, j].set_text_props(fontweight='bold')

ax.set_title('Comparativa CNN vs MLP para datos de imagen', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('images/B05C_fig05.png', dpi=120, bbox_inches='tight')
plt.close()
print('Guardado: images/B05C_fig05.png')

Guardado: images/B05C_fig05.png


## Ejercicio tecnico

Disena un filtro 3x3 que detecte lineas diagonales de arriba-izquierda a abajo-derecha.
Aplicalo a la imagen sintetica creada antes.
En que zonas da respuesta alta? Tiene sentido visual?

TODO: completar el kernel y aplicar la convolucion.

In [8]:
from scipy.ndimage import convolve

# Filtro diagonal principal (arriba-izq a abajo-der)
kernel_diagonal = np.array([
    [ 2, -1, -1],
    [-1,  2, -1],
    [-1, -1,  2]
], dtype=float)

# Aplicar a la imagen
feature_diagonal = convolve(img, kernel_diagonal)
feature_diagonal_relu = np.maximum(0, feature_diagonal)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Imagen original')
axes[1].imshow(feature_diagonal, cmap='RdBu')
axes[1].set_title('Respuesta filtro diagonal')
axes[2].imshow(feature_diagonal_relu, cmap='Blues')
axes[2].set_title('Respuesta tras ReLU')
for ax in axes:
    ax.axis('off')
plt.suptitle('Filtro diagonal: detecta bordes diagonales', fontsize=12)
plt.tight_layout()
plt.savefig('images/B05C_fig05_ejercicio.png', dpi=100, bbox_inches='tight')
plt.show()
print('Guardado: images/B05C_fig05_ejercicio.png')
print()
print('Observacion: el filtro responde donde la intensidad cambia diagonalmente.')
print('Las esquinas de los rectangulos generan respuesta alta al tener cambio diagonal.')

Guardado: images/B05C_fig05_ejercicio.png

Observacion: el filtro responde donde la intensidad cambia diagonalmente.
Las esquinas de los rectangulos generan respuesta alta al tener cambio diagonal.


## Ejercicio de decision

El equipo de calidad quiere clasificar automáticamente documentos escaneados
(facturas, albaranes, contratos) en 5 categorias. Proponen una CNN pre-entrenada (ResNet).

Es la CNN la mejor opcion aqui? Que información necesitas sobre el dataset antes de decidir?
Considerarias un Transformer (ViT) o un modelo de texto (OCR + clasificador)?

**Criterios de evaluacion:**
- Mencionar número de imagenes disponibles (transfer learning cambia el umbral)
- Si la clasificación depende del contenido textual (OCR puede ser mas simple y preciso)
- Si hay variabilidad de diseño visual o tipografia (afecta al poder de la CNN)
- Si el presupuesto computacional permite fine-tuning de un modelo grande
- Si las 5 categorias son visualmente distinguibles o solo por contenido semántico

## Assets generados

- `images/B05C_fig01.png` -- 4 filtros clasicos aplicados a imagen sintetica
- `images/B05C_fig02.png` -- Convolucion paso a paso: ventana deslizandose
- `images/B05C_fig03.png` -- Pipeline: imagen -> Conv -> ReLU -> Pooling
- `images/B05C_fig04.png` -- Encoding CNN: de pixeles a vector abstracto
- `images/B05C_fig05.png` -- Tabla comparativa CNN vs MLP

---